# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
# Limit to single thread to avoid conflicts during hyperparameter tuning
# os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import numpy as np
import optuna
import implicit

from Challenge.paths import load_cv_folds
from Challenge.hyper_tuning import ModelOptimizer
from Challenge.utils import evaluate_recommender_implicit

Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


# **Load Data**

In [4]:
# Load datasets
folds = load_cv_folds(k=5)

In [ ]:
# Initialize optimizer
optimizer = ModelOptimizer("IALS")

## **Hyperparameter search**

In [ ]:
STUDY_NAME = "IALS_implicit_optimization"

In [ ]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "factors": optuna_trial.suggest_categorical("factors", [32, 64, 128]),
        "regularization": optuna_trial.suggest_float("regularization", 0.01, 0.1),
        "alpha": optuna_trial.suggest_float("alpha", 1.0, 40.0),
        "iterations": 15 # Fixed for speed
    }

    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = implicit.cpu.als.AlternatingLeastSquares(
            factors=params["factors"],
            regularization=params["regularization"],
            alpha=params["alpha"],
            iterations=params["iterations"],
            num_threads=0,          # 0 = Use all CPU cores
            random_state=42
        )

        recommender_instance.fit(URM_train)
        
        # Evaluate
        score = evaluate_recommender_implicit(
            recommender_instance,
            20,
            URM_train,
            URM_validation
        )
        
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [7]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=50
)

[I 2025-11-28 13:45:34,862] Using an existing study with name 'IALS_implicit_optimization' instead of creating a new one.


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

Eval Batches: 100%|██████████| 28/28 [00:03<00:00,  8.35it/s]

  Fold 1/5 - Score: 0.24379966545913181


...

Eval Batches: 100%|██████████| 28/28 [00:02<00:00, 11.30it/s]

  Fold 3/5 - Score: 0.22746546470802176


  0%|          | 0/15 [00:00<?, ?it/s]

Eval Batches: 100%|██████████| 28/28 [00:02<00:00, 11.28it/s]


  Fold 4/5 - Score: 0.22634492764273012
[I 2025-11-28 14:05:02,626] Trial 95 finished with value: 0.22691701202414044 and parameters: {'factors': 128, 'regularization': 0.08900303636215784, 'alpha': 39.163882000237464}. Best is trial 68 with value: 0.24322776511452449.

Study statistics: 
  Number of finished trials:  96
  Number of pruned trials:  0
  Number of complete trials:  91

Best Value: 0.24322776511452449
Best Params: {'factors': 128, 'regularization': 0.09986370128000664, 'alpha': 11.650900056856798}


In [14]:
optuna_study.best_value, optuna_study.best_params

(0.24322776511452449,
 {'factors': 128,
  'regularization': 0.09986370128000664,
  'alpha': 11.650900056856798})

In [8]:
optuna.visualization.plot_optimization_history(optuna_study)

In [9]:
optuna.visualization.plot_param_importances(optuna_study)

In [10]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Extend ranges**

In [15]:
STUDY_NAME = "IALS_implicit_optimization_v2"

In [16]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "factors": optuna_trial.suggest_int("factors", 128, 512, step=32),
        "regularization": optuna_trial.suggest_float("regularization", 0.01, 1.0, log=True),
        "alpha": optuna_trial.suggest_float("alpha", 5.0, 20.0),
        "iterations": 15, # Fixed for speed
        "num_threads": 0  # Use all CPU cores
    }

    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = implicit.cpu.als.AlternatingLeastSquares(
            factors=params["factors"],
            regularization=params["regularization"],
            alpha=params["alpha"],
            iterations=params["iterations"],
            num_threads=params["num_threads"],         
            random_state=42
        )

        recommender_instance.fit(URM_train, show_progress=False)
        
        # Evaluate
        score = evaluate_recommender_implicit(
            recommender_instance,
            20,
            URM_train,
            URM_validation
        )
        
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [ ]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=50
)

[I 2025-11-28 14:24:44,846] A new study created in RDB with name: IALS_implicit_optimization_v2


  0%|          | 0/50 [00:00<?, ?it/s]

Eval Batches: 100%|██████████| 28/28 [00:02<00:00,  9.50it/s]


  Fold 1/5 - Score: 0.21233080132359572


Eval Batches: 100%|██████████| 28/28 [00:02<00:00, 10.08it/s]


...

  Fold 1/5 - Score: 0.20221817627520078


Eval Batches: 100%|██████████| 28/28 [00:02<00:00,  9.79it/s]


  Fold 2/5 - Score: 0.20284415986629928


Eval Batches: 100%|██████████| 28/28 [00:02<00:00,  9.77it/s]


  Fold 3/5 - Score: 0.20254688652922662


In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- (0.24322776511452449,
 {'factors': 128,
  'regularization': 0.09986370128000664,
  'alpha': 11.650900056856798})